# Train — MoCo v2 (GPU 0)

**실행 환경**: 이 노트북은 GPU 0이 할당된 Jupyter 인스턴스에서 실행한다.

터미널에서 다음과 같이 노트북 서버 시작:
```bash
CUDA_VISIBLE_DEVICES=0 jupyter lab --port 8888
```

이러면 이 노트북에서는 `torch.cuda.device_count() == 1`이 되고,
다른 GPU는 보이지도 않는다 (실수 방지).

**병렬 학습**: 동시에 별도 터미널에서 `CUDA_VISIBLE_DEVICES=1 jupyter lab --port 8889`로
병행 학습이 필요하면 다른 GPU에서 train_mocov3 noteboook 실행.

## Cell 1 — 환경 확인

In [ ]:
%load_ext autoreload
%autoreload 2

import torch

assert torch.cuda.is_available(), 'GPU 사용 불가!'
assert torch.cuda.device_count() == 1, (
    f'GPU {torch.cuda.device_count()}개 보임. '
    f'CUDA_VISIBLE_DEVICES=0으로 노트북 띄웠는지 확인.'
)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Config 로드 + 환경별 override

In [ ]:
import yaml
from pathlib import Path

with open('../configs/mocov2_r50.yaml') as f:
    cfg = yaml.safe_load(f)

# 경로를 노트북 기준 상대경로로 보정
cfg['data']['root'] = '../data'
cfg['output']['dir'] = '../outputs/mocov2_r50_seed42'
cfg['output']['log_file'] = '../logs/mocov2_seed42.log'

# 처음 sanity check할 때는 epoch 줄여서 빠르게
# cfg['schedule']['epochs'] = 5
# cfg['training']['batch_size'] = 256

print(yaml.dump(cfg, allow_unicode=True))

## Cell 3 — 학습 시작

이 셀이 가장 오래 돈다 (400 epoch ≈ 20~24시간 예상).

**주의**:
- 브라우저 닫혀도 커널은 살아있긴 한데, 진행상황 stream은 끊김.
- 안전을 위해 `../logs/mocov2_seed42.log`에도 같은 로그가 기록됨.
- 백그라운드 안전 실행은 `bash ../scripts/run_mocov2.sh` 사용.

In [ ]:
from ssl_lib.train_loop import pretrain

pretrain(cfg)

## (선택) Resume 학습

학습이 도중에 죽으면 마지막 체크포인트에서 이어서:

In [ ]:
# pretrain(cfg, resume_from='../outputs/mocov2_r50_seed42/ckpt_ep200.pth')

## (디버그) 한 epoch만 돌려보기

In [ ]:
# from ssl_lib.train_loop import build_model, train_one_epoch
# from ssl_lib.utils import set_seed, build_optimizer, CosineLRScheduler, setup_logger
# from ssl_lib.data import build_stl10_loader
# 
# set_seed(cfg['training']['seed'])
# device = torch.device('cuda')
# logger = setup_logger('debug', log_file=None)
# 
# loader = build_stl10_loader(cfg)
# model = build_model(cfg).to(device)
# optimizer = build_optimizer(model, cfg)
# lr_sched = CosineLRScheduler(optimizer, base_lr=cfg['optimizer']['lr'],
#                              warmup_steps=len(loader)*cfg['schedule']['warmup_epochs'],
#                              total_steps=len(loader)*cfg['schedule']['epochs'])
# scaler = torch.cuda.amp.GradScaler() if cfg['training']['amp'] else None
# 
# stats = train_one_epoch(
#     model=model, loader=loader, optimizer=optimizer, lr_scheduler=lr_sched,
#     epoch=0, total_epochs=cfg['schedule']['epochs'], device=device,
#     scaler=scaler, log_every=10, logger=logger, momentum_ema=False,
# )
# print(stats)